In [1]:
# Check GPU Type
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Select the Runtime > "Change runtime type" menu to enable a GPU accelerator, ')
    print('and then re-execute this cell.')
else:
    print(gpu_info)

Mon Oct 13 00:07:05 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 571.96                 Driver Version: 571.96         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1060 3GB  WDDM  |   00000000:01:00.0  On |                  N/A |
| 35%   37C    P8              8W /  120W |     500MiB /   3072MiB |      7%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import time, random, math, enum
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import cv2
import matplotlib.pyplot as plt

import albumentations as albu

import tensorflow as tf

import torch
import torch.nn as nn
import torchvision


print("Tensorflow version " + tf.__version__)
print("torch version " + torch.__version__)
print("torchvision version " + torchvision.__version__)
print("albumentations version " + albu.__version__)

Tensorflow version 2.18.0
torch version 2.7.1+cu126
torchvision version 0.22.1+cu126
albumentations version 2.0.8


In [3]:
train_df = pd.read_csv('F:/ml_data_resource/kaggle/intracranial_aneurysm_detection/train.csv')
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4348 entries, 0 to 4347
Data columns (total 18 columns):
 #   Column                                      Non-Null Count  Dtype 
---  ------                                      --------------  ----- 
 0   SeriesInstanceUID                           4348 non-null   object
 1   PatientAge                                  4348 non-null   int64 
 2   PatientSex                                  4348 non-null   object
 3   Modality                                    4348 non-null   object
 4   Left Infraclinoid Internal Carotid Artery   4348 non-null   int64 
 5   Right Infraclinoid Internal Carotid Artery  4348 non-null   int64 
 6   Left Supraclinoid Internal Carotid Artery   4348 non-null   int64 
 7   Right Supraclinoid Internal Carotid Artery  4348 non-null   int64 
 8   Left Middle Cerebral Artery                 4348 non-null   int64 
 9   Right Middle Cerebral Artery                4348 non-null   int64 
 10  Anterior Communicating A

In [4]:
train_df.head(15)

,SeriesInstanceUID,PatientAge,PatientSex,Modality,Left Infraclinoid Internal Carotid Artery,Right Infraclinoid Internal Carotid Artery,Left Supraclinoid Internal Carotid Artery,Right Supraclinoid Internal Carotid Artery,Left Middle Cerebral Artery,Right Middle Cerebral Artery,Anterior Communicating Artery,Left Anterior Cerebral Artery,Right Anterior Cerebral Artery,Left Posterior Communicating Artery,Right Posterior Communicating Artery,Basilar Tip,Other Posterior Circulation,Aneurysm Present
0,1.2.826.0.1.3680043.8.498.10004044428023505108...,64,Female,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1.2.826.0.1.3680043.8.498.10004684224894397679...,76,Female,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1.2.826.0.1.3680043.8.498.10005158603912009425...,58,Male,CTA,0,0,0,0,0,0,0,0,0,0,0,0,1,1
3,1.2.826.0.1.3680043.8.498.10009383108068795488...,71,Male,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1.2.826.0.1.3680043.8.498.10012790035410518400...,48,Female,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,1.2.826.0.1.3680043.8.498.10014757658335054766...,53,Female,CTA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,1.2.826.0.1.3680043.8.498.10021411248005513321...,55,Female,CTA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,1.2.826.0.1.3680043.8.498.10022688097731894079...,51,Female,MRA,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,1.2.826.0.1.3680043.8.498.10022796280698534221...,78,Male,CTA,0,0,0,0,0,1,0,0,0,0,0,0,0,1
9,1.2.826.0.1.3680043.8.498.10023411164590664678...,76,Female,MRA,0,0,0,0,0,1,0,0,0,0,0,0,0,1


In [47]:
torchvision.models.MobileNetV2.__name__.lower()

'mobilenetv2'

In [48]:
mobilenet = torchvision.models.mobilenet_v2(weights=torchvision.models.MobileNet_V2_Weights.DEFAULT)
mobilenet

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [6]:
'resnet_{0:02d}_test'.format(2)

'resnet_02_test'

In [9]:
time.strftime('%Y%m%d_%H%M%S')

'20251003_173938'

In [10]:
y_true = np.array([0., 0., 0., 0., 1.,
                   0., 0., 0., 0., 0.,
                   0., 0., 0., 1.,])

y_pred = np.array([0., 0.5, 0., 0., 0.4,
                   0., 0.2, 0.1, 0.2, 0.,
                   0., 0.3, 0., 0.7,])

In [31]:
y2_true = np.array([[0., 0., 0., 0., 1.,
                   0., 0., 0., 0., 0.,
                   0., 0., 0., 1.,],
                   [0., 0., 0., 0., 0.,
                   0., 0., 0., 0., 0.,
                   0., 0., 0., 0.,]])

y2_pred = np.array([[0., 0.5, 0., 0., 0.4,
                   0., 0.2, 0.1, 0.2, 0.,
                   0., 0.3, 0., 0.7,],
                   [0., 0.5, 0., 0.8, 0.4,
                   0., 0., 0.1, 0.2, 0.,
                   0., 0.3, 0., 0.4,]])

In [32]:
y2_true.shape

(2, 14)

In [33]:
y2_pred.shape

(2, 14)

In [11]:
threshold_middle = 0.5
threshold_margin = 0.0625
threshold_upper = min(threshold_middle + threshold_margin, 1)
threshold_lower = max(threshold_middle - threshold_margin, 0)
threshold_lower, threshold_middle, threshold_upper

(0.4375, 0.5, 0.5625)

In [62]:
auc_weights = [1., 1., 1., 1., 1.,
               1., 1., 1., 1., 1.,
               1., 1., 1., 13.,]

auc_weights = auc_weights / np.sum(auc_weights)
auc_weights

array([0.03846154, 0.03846154, 0.03846154, 0.03846154, 0.03846154,
       0.03846154, 0.03846154, 0.03846154, 0.03846154, 0.03846154,
       0.03846154, 0.03846154, 0.03846154, 0.5       ])

In [63]:
auc_weights = np.expand_dims(auc_weights, axis=0)
auc_weights

array([[0.03846154, 0.03846154, 0.03846154, 0.03846154, 0.03846154,
        0.03846154, 0.03846154, 0.03846154, 0.03846154, 0.03846154,
        0.03846154, 0.03846154, 0.03846154, 0.5       ]])

In [65]:
auc_weights = np.repeat(auc_weights, 2, axis=0)

In [66]:
auc_weights

array([[0.03846154, 0.03846154, 0.03846154, 0.03846154, 0.03846154,
        0.03846154, 0.03846154, 0.03846154, 0.03846154, 0.03846154,
        0.03846154, 0.03846154, 0.03846154, 0.5       ],
       [0.03846154, 0.03846154, 0.03846154, 0.03846154, 0.03846154,
        0.03846154, 0.03846154, 0.03846154, 0.03846154, 0.03846154,
        0.03846154, 0.03846154, 0.03846154, 0.5       ]])

In [68]:
y2_pred.shape

(2, 14)

In [69]:
auc_weights.shape

(2, 14)

In [73]:
np.where(y2_pred >= threshold_upper, auc_weights, 0)

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.5       ],
       [0.        , 0.        , 0.        , 0.03846154, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ]])

In [74]:
np.where(y2_true >= 1.0, 1, 0)

array([[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [ ]:
np.where(y2_pred >= threshold_upper, auc_weights, 0) * np.where(y2_true >= 1.0, 1, 0)

array([[0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
        0.5],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
        0. ]])

In [81]:
valid_weights = np.sum(np.where(y2_pred >= threshold_upper, auc_weights, 0) * np.where(y2_true >= 1.0, 1, 0))
valid_weights

0.5

In [80]:
np.sum(valid_weights)

0.5

In [71]:
np.sum(np.where(y2_pred >= threshold_upper, auc_weights, 0))

0.5384615384615384

In [44]:
positive_pred = np.where(y2_pred >= threshold_upper)
positive_pred

(array([0, 1], dtype=int64), array([13,  3], dtype=int64))

In [45]:
np.where(y2_true[positive_pred] >= 1.0)

(array([0], dtype=int64),)

In [54]:
len(np.where(y2_true[positive_pred] >= 1.0)[-1])

1

In [46]:
negative_pred = np.where(y2_pred <= threshold_lower)
negative_pred

(array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1], dtype=int64),
 array([ 0,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12,  0,  2,  4,  5,  6,
         7,  8,  9, 10, 11, 12, 13], dtype=int64))

In [47]:
y2_true[negative_pred]

array([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0.])

In [53]:
len(np.where(y2_true[negative_pred] <= 0.0)[-1])

23

In [5]:
train_localizers_df = pd.read_csv('F:/ml_data_resource/kaggle/intracranial_aneurysm_detection/train_localizers.csv')
train_localizers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2254 entries, 0 to 2253
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   SeriesInstanceUID  2254 non-null   object
 1   SOPInstanceUID     2254 non-null   object
 2   coordinates        2254 non-null   object
 3   location           2254 non-null   object
dtypes: object(4)
memory usage: 70.6+ KB


In [6]:
train_localizers_df.head(12)

,SeriesInstanceUID,SOPInstanceUID,coordinates,location
0,1.2.826.0.1.3680043.8.498.10005158603912009425...,1.2.826.0.1.3680043.8.498.10775329348174902199...,"{'x': 258.3621186176837, 'y': 261.359900373599}",Other Posterior Circulation
1,1.2.826.0.1.3680043.8.498.10022796280698534221...,1.2.826.0.1.3680043.8.498.53868409774237283281...,"{'x': 194.87253141831238, 'y': 178.32675044883...",Right Middle Cerebral Artery
2,1.2.826.0.1.3680043.8.498.10023411164590664678...,1.2.826.0.1.3680043.8.498.24186535344744886473...,"{'x': 189.23979878597123, 'y': 209.19184886465...",Right Middle Cerebral Artery
3,1.2.826.0.1.3680043.8.498.10030095840917973694...,1.2.826.0.1.3680043.8.498.75217084841854214544...,"{'x': 208.2805049088359, 'y': 229.78962131837307}",Right Infraclinoid Internal Carotid Artery
4,1.2.826.0.1.3680043.8.498.10034081836061566510...,1.2.826.0.1.3680043.8.498.71237104731452368587...,"{'x': 249.86745590416498, 'y': 220.623044646393}",Anterior Communicating Artery
5,1.2.826.0.1.3680043.8.498.10035643165968342618...,1.2.826.0.1.3680043.8.498.30083322134992576720...,"{'x': 223.66020358711827, 'y': 225.3447011195274}",Right Anterior Cerebral Artery
6,1.2.826.0.1.3680043.8.498.10035643165968342618...,1.2.826.0.1.3680043.8.498.46752468449107005352...,"{'x': 289.2376764288231, 'y': 211.78100912436958}",Left Middle Cerebral Artery
7,1.2.826.0.1.3680043.8.498.10035643165968342618...,1.2.826.0.1.3680043.8.498.14504961303532677815...,"{'x': 232.88987039309842, 'y': 226.4305401310092}",Right Supraclinoid Internal Carotid Artery
8,1.2.826.0.1.3680043.8.498.10042423585566957032...,1.2.826.0.1.3680043.8.498.63062558671948377310...,"{'x': 140.1086940862857, 'y': 177.63838968827815}",Right Middle Cerebral Artery
9,1.2.826.0.1.3680043.8.498.10042474696169267476...,1.2.826.0.1.3680043.8.498.89290443797896401250...,"{'x': 254.25852272727272, 'y': 221.56586270871...",Left Supraclinoid Internal Carotid Artery


In [ ]:
import SimpleITK as sitk
import matplotlib.pyplot as plt
import ipywidgets as widgets

In [ ]:
# 이미지 파일을 읽어옵니다.
dicom_image = sitk.ReadImage('F:/ml_data_resource/kaggle/intracranial_aneurysm_detection/series_debug/1.2.826.0.1.3680043.8.498.10102361048562788202568222767625052953/1.2.826.0.1.3680043.8.498.36001130720722571820822507347231024664.dcm')

# 1. 원점 가져오기
origin = dicom_image.GetOrigin()  # dicom_image의 원점을 반환합니다.
print("원점:", origin) # 원점: (-142.448, -161.0, -682.743)

# 2. 크기 가져오기
size = dicom_image.GetSize()  # dicom_image의 크기를 반환합니다.
print("크기:", size) # 크기: (512, 512, 1)

# 3. 깊이 가져오기
depth = dicom_image.GetDepth()  # dicom_image의 깊이(마지막 차원)를 반환합니다.
print("깊이:", depth) # 깊이: 1

# 4. 특정 위치의 픽셀 값 가져오기
pixel_location = (0, 0, 0)  # 예시로 (0, 0, 0) 위치를 지정합니다.
pixel_value = dicom_image.GetPixel(*pixel_location)  # pixel_location 위치의 픽셀 값을 반환합니다.
print("픽셀 값 (0, 0, 0):", pixel_value) #픽셀 값 (0, 0, 0): -2048

# 5. 원점 설정하기
new_origin = [0.0, 0.0, 0.0]  # 새 원점 값을 정의합니다.
dicom_image.SetOrigin(new_origin)  # dicom_image의 원점을 설정합니다.
print("새 원점 설정 완료:", new_origin)  #새 원점 설정 완료: [0.0, 0.0, 0.0]

In [ ]:
def load_dicom_series(dcm_folder_path):
    """
    DICOM 폴더 내의 모든 DICOM 파일을 3D 이미지로 읽기
    """
    reader = sitk.ImageSeriesReader()  # SimpleITK의 ImageSeriesReader 객체를 생성
    dicom_series = reader.GetGDCMSeriesFileNames(dcm_folder_path)  # 폴더 내의 모든 DICOM 파일들의 이름 추출
    reader.SetFileNames(dicom_series)  # 읽어올 DICOM 파일들의 이름을 설정
    dicom_images = reader.Execute()  # DICOM 파일들을 읽어서 3D 이미지로 생성
    return dicom_images  # 3D 이미지를 반환

def dicom_to_numpy(dicom_images):
    """
    DICOM 이미지를 3D NumPy 배열로 변환
    """
    return sitk.GetArrayFromImage(dicom_images)  # SimpleITK 이미지를 NumPy 배열로 변환하여 반환

def display_image(array):
    """
    3D NumPy 배열을 슬라이서 위젯을 사용하여 표시
    """
    # 슬라이드를 사용하여 슬라이스를 스크롤
    def view_image(slice_index):
        plt.figure(figsize=(10, 10))  # 이미지를 표시할 Figure를 설정
        plt.imshow(array[slice_index], cmap='gray')  # 현재 슬라이스 이미지를 회색조로 표시
        plt.title(f'Slice {slice_index}')  # 슬라이스 번호를 제목으로 설정
        plt.show()  # 이미지를 출력

    slice_slider = widgets.IntSlider(min=0, max=array.shape[0] - 1, step=1, description='Slice:')  # 슬라이더를 생성
    widgets.interact(view_image, slice_index=slice_slider)  # 슬라이더와 view_image 함수를 연결하여 상호작용

In [ ]:
# DICOM 폴더 내의 모든 DICOM 파일을 3D 이미지로 읽기
# dicom_images = load_dicom_series('F:/ml_data_resource/kaggle/intracranial_aneurysm_detection/series_debug/1.2.826.0.1.3680043.8.498.10102361048562788202568222767625052953/')
dicom_images = load_dicom_series('F:/ml_data_resource/kaggle/intracranial_aneurysm_detection/series/1.2.826.0.1.3680043.8.498.10012790035410518400400834395242853657/')

# DICOM 이미지를 3D NumPy 배열로 변환
dicom_array = dicom_to_numpy(dicom_images)

# 3D NumPy 배열을 표시
display_image(dicom_array)

In [ ]:
!python drill_and_debug.py